## BiGG overwrite + first manual mass/charge balance curation

## Imports

In [ ]:
import cobra
from cobra.io import read_sbml_model, write_sbml_model
from cobra.manipulation.validate import check_mass_balance
from cobra import Metabolite
import pandas as pd
import os
import ast
import logging
import warnings

import sys
sys.path.insert(0, '/home/emma/Dokumente/thesis')

from functions import *

## Paths

In [14]:
# Path to the draft models
model_dir = '/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/faa_models/'

save_dir = '/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/'


## Overwrite metabolite and reactions info with current BiGG database data

### BiGG infos

In [6]:
bigg_metabolites = pd.read_csv('/home/emma/Dokumente/thesis/Model_generation_curation/bigg_metabolites_complete.csv')

In [7]:
bigg_reactions = pd.read_csv('/home/emma/Dokumente/thesis/Model_generation_curation/bigg_reactions_complete.csv')

In [5]:
# manually add fdxox_c as a metabolite
fdxox_c = Metabolite(
    'fdxox_c',
    formula='Fe2S2',
    name='Oxidized ferredoxin',
    compartment='c')

### Functions

In [6]:
def overwrite_with_BiGG_metabolites(model):

    for metabolite in model.metabolites:
        
        bigg_formula_list = ast.literal_eval(bigg_metabolites.loc[bigg_metabolites['bigg_id'] == metabolite.id[:-2], 'formulae'].values[0])
        bigg_charge_list = ast.literal_eval(bigg_metabolites.loc[bigg_metabolites['bigg_id'] == metabolite.id[:-2], 'charges'].values[0])

        if len(bigg_formula_list) == 1:
            old_formula = metabolite.formula
            if old_formula != bigg_formula_list[0]:
                metabolite.formula = bigg_formula_list[0]
        if len(bigg_charge_list) == 1:
            old_charge = metabolite.charge
            if old_charge != int(bigg_charge_list[0]):
                metabolite.charge = int(bigg_charge_list[0])


In [7]:
# checks the mass and charge balance for every reaction in a model
def check_balance(model, print_results=True):
    unbalanced_reactions = check_mass_balance(model)
    if print_results:
        print("There are {0} unbalanced reactions in {1}".format(len(unbalanced_reactions), model) )
    return unbalanced_reactions


In [8]:
def overwrite_with_BIGG_reactions(model):
    unbalanced_rxns = check_balance(model, print_results=False)
    unbalanced_rxns = [r.id for r in unbalanced_rxns]

    for rxn in unbalanced_rxns:

        new_react = ast.literal_eval(bigg_reactions[bigg_reactions['bigg_id'] == rxn]["equation"].iloc[0])

        new_mets_dict = {model.metabolites.get_by_id(met_id): coeff for met_id, coeff in new_react.items()}

        reaction = model.reactions.get_by_id(rxn)
        reaction.subtract_metabolites(reaction.metabolites)
        reaction.add_metabolites(new_mets_dict)


## Manual curation (Script from Lisa)

In [26]:
# manually add fdxox_c as a metabolite
fdxox_c = Metabolite(
    'fdxox_c',
    formula='Fe2S2',
    name='Oxidized ferredoxin',
    compartment='c')

m2butp_c = Metabolite(
    'm2butp_c',
    formula='C5H9O5P',
    name='2-methylbutanoyl-phosphate',
    compartment='c')


lgt__S_c = Metabolite(
    'lgt__S_c',
    formula='C13H20N3O8S',
    name='(R)-S-Lactoylglutathione',
    compartment='c')


manglyc_p = Metabolite(
    'manglyc_p',
    formula='C9H15O9',
    name='2(alpha-D-Mannosyl)-D-glycerate',
    compartment='p')

### Functions

In [9]:
def overwrite_charge(model, rxn_id, new_charge):
    if rxn_id in model.metabolites:
        met = model.metabolites.get_by_id(rxn_id)
        old_charge = met.charge
        met.charge = new_charge


In [10]:
def overwrite_formula(model, rxn_id, new_formula):
    if rxn_id in model.metabolites:
        met = model.metabolites.get_by_id(rxn_id)
        old_formula = met.formula
        met.formula = new_formula


In [11]:
def overwrite_reaction(model, rxn_id, new_rxn_dict):
    if rxn_id in model.reactions:
        try:
            rxn = model.reactions.get_by_id(rxn_id)
            old_metabolites = {met.id: coeff for met, coeff in rxn.metabolites.items()}
            rxn.subtract_metabolites(rxn.metabolites)
            rxn.add_metabolites(new_rxn_dict)
            new_metabolites = {met.id: coeff for met, coeff in rxn.metabolites.items()}
        except KeyError:
            print(f"{model} does not contain one of the metabolites")

In [12]:
def delete_metabolite(model, met_id):
    if met_id in model.metabolites:

        if len(model.metabolites.get_by_id(met_id).reactions) == 0:
            met = model.metabolites.get_by_id(met_id)
            old_formula = met.formula
            old_charge = met.charge
            model.metabolites.remove(met)

        else:
            print(f'metabolite {met_id} cannot be deleted from {model.id} because of reaction(s): {model.metabolites.get_by_id(met_id).reactions}')

In [13]:
def delete_reaction(model, rxn_id):
    if rxn_id in model.reactions:
        rxn = model.reactions.get_by_id(rxn_id)
        old_metabolites = {met.id: coeff for met, coeff in rxn.metabolites.items()}
        model.remove_reactions([rxn])

In [14]:
def delete_lonely_reaction(model, rxn_id):
    # in comparison to the above function, this function is meant for reactions that are only deleted because all participating metabolites are only in this reaction;
    # the delete_reaction function will always delete a reaction;
    # currently to be on the safe side, we will only delete reactions if all metabolites have only this reaction
    if rxn_id in model.reactions:
        rxn = model.reactions.get_by_id(rxn_id)
        mets = rxn.metabolites
        dead_rxn = [met for met in mets if len(mets.reactions) == 1]

        if len(dead_rxn) == len(mets):
            model.remove_reactions([rxn])
            for met in dead_rxn:
                delete_metabolite(model, met.id)
            print(f"Reaction '{rxn_id}' and all its metabolites were removed.")
        elif dead_rxn:
            print(f"⚠️ Reaction '{rxn_id}' has some unique metabolites:")
            for met in dead_rxn:
                print(f"   - {met.id}")
            print("Please check manually before deleting.")

In [15]:
def add_metabolites(model, rxn_id, new_met_dict): #introduced by emma 
    #Manually adds metabolites to a specific reaction in a model.
    #Accepts both  Metabolite objects or string IDs as dictionary keys.
    if rxn_id not in model.reactions:
        return 
        
    rxn = model.reactions.get_by_id(rxn_id)
    model_specific_mets = {}
    
    for met, coeff in new_met_dict.items():
        # string id 
        if isinstance(met, str):
            met_id = met
            if met_id in model.metabolites:
                local_met = model.metabolites.get_by_id(met_id)
            else:
                local_met = Metabolite(met_id)
                model.add_metabolites([local_met])
        
        # metabolite object
        else:
            met_id = met.id
            if met_id in model.metabolites:
                local_met = model.metabolites.get_by_id(met_id)
            else:
                local_met = met.copy()
        
        model_specific_mets[local_met] = coeff

    # does reaction needs updating?
    needs_update = False
    for met, coeff in model_specific_mets.items():
        if met not in rxn.metabolites or rxn.metabolites[met] != coeff:
            needs_update = True
            break
    
    if needs_update:
        rxn.add_metabolites(model_specific_mets)
        print(f"Successfully updated reaction {rxn_id} in model {model.id}")


In [ ]:
def overwrite_manual(model):
    # first all changes to metabolites, i.e. charges and formulas
    # afterwards changes for reactions, i.e. changing stoichiometry, replacing/deleting metabolites (especially H)
    # last deletions (mostly reactions if duplicate but also metabolites)
    # every category is alphabetically sorted

    # first: metabolites
    overwrite_formula(model, "2ameph_p", "C2H7NO3P") # og = C2H8NO3P, charge was changed from 0 to -1 automatically with bigg and formula now also needed to be changed
    overwrite_formula(model, "2ameph_e", "C2H7NO3P")
    overwrite_formula(model, "2ameph_c", "C2H7NO3P")
    overwrite_charge(model, "2dhphaccoa_c", -4) #og = 0; according to seed https://modelseed.org/biochem/compounds/cpd16740
    overwrite_charge(model, "2mpdhl_c", -1)
    overwrite_charge(model, "23dhbzs3_c", -1) # og = 0, -1 alterative in bigg
    overwrite_formula(model, "3hsa_c", "C19H24O3")
    overwrite_charge(model, "3sala_c", -1) # og = 0
    overwrite_formula(model, "3sala_c", "C3H6NO4S")
    overwrite_formula(model, "34dhsa_c", "C19H24O4")
    overwrite_charge(model, "4cml_c", -2)
    overwrite_formula(model, "4cml_c", "C7H4O6")
    overwrite_charge(model, "4hoxpac_c", -1) # og = 0, -1 alterative in bigg
    overwrite_charge(model, "4hoxpac_e", -1)
    overwrite_charge(model, "4hoxpac_p", -1)
    overwrite_formula(model, "49dsha_c", "C19H23O6")
    overwrite_charge(model, "49dsha_c", -1)
    overwrite_charge(model, "5aizc_c", -3)
    overwrite_formula(model, "5ohhipcoa_c", "C34H50N7O19P3S")
    overwrite_charge(model, "5ohhipcoa_c", -4)
    overwrite_charge(model, "6pgg_c", -2) # og = 0, -2 alterative in bigg and in accordance with ecoli
    overwrite_formula(model, "9ohadd_c", "C19H24O3")

    overwrite_charge(model, "aad_c", -2)
    overwrite_formula(model, "abg4_c", "C12H12N2O5") # C12H11N2O5; https://biocyc.org/compound?orgid=META&id=CPD0-889
    overwrite_charge(model, "abg4_c", -2) # og = 0
    overwrite_formula(model, "abg4_e", "C12H12N2O5")
    overwrite_charge(model, "abg4_e", -2)
    overwrite_formula(model, "acadl_c", "C12H14N5O8P") # og = C12H14N5O8P; https://pubchem.ncbi.nlm.nih.gov/compound/440867 with h16 charge = 0, we have -1 charge
    overwrite_charge(model, "acadl_c", -2) # https://pubchem.ncbi.nlm.nih.gov/compound/440867 with h16 is charge = 0 and we have H14, so we need -2
    overwrite_charge(model, "ACP_c", 0)
    overwrite_charge(model, "actACP_c", -1)
    overwrite_charge(model, "acysbmn_e", -1) # og = 0, -1 according to metacyc with same formula https://metacyc.org/compound?orgid=META&id=CPD1G-185
    overwrite_charge(model, "acysbmn_c", -1) # og = 0, -1
    overwrite_charge(model, "ah6p__D_c", -2) # og = 0, -2 must be because f6p is also -2 and they can directly converted into each other
    overwrite_charge(model, "air_c", -2)
    overwrite_charge(model, "amacald_c", 1)
    overwrite_formula(model, "amacald_c", "C2H6NO")
    overwrite_formula(model, "andrs14dn317dn_c", "C19H24O2")
    overwrite_formula(model, "apoACP_c", "C373H582N94O136S2") # og = C373H583N94O136S2; charge was changed from 1 to 0 and now the amount of H also reflects that
    overwrite_charge(model, "aso3_c", -1)
    overwrite_charge(model, "aso3_e", -1)
    overwrite_charge(model, "aso3_p", -1)
    overwrite_formula(model, "aso3_c", "H2O3As")
    overwrite_formula(model, "aso3_e", "H2O3As")
    overwrite_formula(model, "aso3_p", "H2O3As")
    overwrite_formula(model, "aso4_c", "HO4As")
    overwrite_formula(model, "aso4_e", "HO4As")
    overwrite_formula(model, "aso4_p", "HO4As")

    overwrite_charge(model, "bmn_c", 2) # og = 0, to balance BMNMSHS (bmn is just imported for this reaction)
    overwrite_charge(model, "bmn_e", 2)
    overwrite_charge(model, "but2eACP_c", -1)

    overwrite_charge(model, "CCbuttc_c", -3) # og = -3; to balance reaction with 4cml_c
    overwrite_formula(model, "CCbuttc_c", "C7H3O6") # C7H3O6; to reflect charge change
    overwrite_formula(model, "cchol_c", "C27H42O3")
    overwrite_charge(model, "cdigmp_c", -2)
    overwrite_formula(model, "cholc3coa_c", "C43H66N7O18P3S")
    overwrite_formula(model, "cholc5coa_c", "C45H70N7O18P3S")
    overwrite_formula(model, "cholc8coa_c", "C48H76N7O18P3S")
    overwrite_formula(model, "cholenec3coa_c", "C43H64N7O18P3S")
    overwrite_formula(model, "cholenec5coa_c", "C45H68N7O18P3S")
    overwrite_formula(model, "cholenec8coa_c", "C48H74N7O18P3S")

    overwrite_formula(model, "decoa_c", "C31H48N7O17P3S")
    overwrite_charge(model, "dgal6p_c", -2)
    overwrite_charge(model, "dmlgnc_c", -1) # og = 0; https://modelseed.org/biochem/compounds/cpd15951
    overwrite_charge(model, "dtbt_c", -1)

    overwrite_charge(model, "fad_c", -2)
    overwrite_charge(model, "fad_e", -2)
    overwrite_charge(model, "fad_p", -2)
    overwrite_charge(model, "fe3dhbzs3_c", 3) # og=0, alternative in bigg
    overwrite_formula(model, "fe3dhbzs3_c", "C30FeH29N3O16") # og = C30FeH28N3O16, with H29 is in bigg and ecoli
    overwrite_formula(model, "fe3dhbzs3_e", "C30FeH29N3O16")
    overwrite_formula(model, "fe3dhbzs3_p", "C30FeH29N3O16")
    overwrite_formula(model, "feoxam_c", "C25H46FeN6O8") # formula change according to bigg and ecoli
    overwrite_formula(model, "feoxam_e", "C25H46FeN6O8")
    overwrite_formula(model, "feoxam_p", "C25H46FeN6O8")
    overwrite_charge(model, "ficytc_c", 1)
    overwrite_charge(model, "fmcbtt_c", 2) # og = 0 but it has fe2 in it
    overwrite_charge(model, "fmn_c", -2)
    overwrite_formula(model, "fmn_c", "C17H19N4O9P")
    overwrite_charge(model, "fmn_e", -2)
    overwrite_formula(model, "fmn_e", "C17H19N4O9P")
    overwrite_charge(model, "fmn_p", -2)
    overwrite_formula(model, "fmn_p", "C17H19N4O9P")
    overwrite_charge(model, "focytc_c", 1)
    overwrite_charge(model, "fpram_c", -1)
    overwrite_formula(model, "fpram_c", "C8H15N3O8P")

    overwrite_charge(model, "g3p_c", -2)
    overwrite_charge(model, "g6p_A_c", -2)
    overwrite_formula(model, "galam6p_c", "C6H13NO8P") # https://biocyc.org/compound?orgid=META&id=D-GALACTOSAMINE-6-PHOSPHATE
    overwrite_charge(model, "galam6p_c", -1)
    overwrite_charge(model, "galam_p", +1) #og = 0 https://metacyc.org/compound?orgid=META&id=GALACTOSAMINE
    overwrite_formula(model, "galam_p", "C6H14NO5")
    overwrite_charge(model, "galam_e", +1)
    overwrite_formula(model, "galam_e", "C6H14NO5")
    overwrite_charge(model, "gcvHL_ADPr_c", -1)
    overwrite_formula(model, "gcvHL_ADPr_c", "C23H36N6O21P4S2")
    overwrite_charge(model, "gcvHL_nhLA_c", 0)
    overwrite_formula(model, "gcvHL_nhLA_c", "C8H16NO8P2S2")
    overwrite_charge(model, "gdptp_c", -7)
    overwrite_charge(model, "glutrna_c", -3)
    overwrite_formula(model, "glycogen_c", "C6H10O5")
    overwrite_charge(model, "gly_pro__L_c", 1)
    overwrite_formula(model, "gly_pro__L_c", "C7H13N2O3")
    overwrite_charge(model, "gly_pro__L_e", 1)
    overwrite_formula(model, "gly_pro__L_e", "C7H13N2O3")
    overwrite_formula(model, "gly_tyr_c", "C11H14N2O4")
    overwrite_formula(model, "gly_phe_c", "C11H14N2O3")
    overwrite_formula(model, "gly_leu_c", "C8H16N2O3")
    overwrite_formula(model, "gly_cys_c", "C5H10N2O3S")

    overwrite_formula(model, "hchol_c", "C27H44O2")
    overwrite_formula(model, "hcholc8coa_c", "C48H76N7O19P3S")
    overwrite_formula(model, "hcholc5coa_c", "C45H70N7O19P3S")
    overwrite_formula(model, "hcholc3coa_c", "C43H66N7O19P3S")
    overwrite_charge(model, "hethmpp_c", -2)
    overwrite_charge(model, "hemeO_c", -2)
    overwrite_formula(model, "hia_c", "C11H16O4")
    overwrite_formula(model, "hia_e", "C11H16O4")
    overwrite_formula(model, "hip_c", "C13H17O4")
    overwrite_charge(model, "hip_c", -1)
    overwrite_formula(model, "hipcoa_c", "C34H48N7O19P3S")
    overwrite_charge(model, "hipcoa_c", -4)
    overwrite_formula(model, "hipecoa_c", "C34H48N7O19P3S")
    overwrite_charge(model, "hipecoa_c", -4)
    overwrite_formula(model, "hipohcoa_c", "C34H50N7O20P3S")
    overwrite_charge(model, "hipohcoa_c", -4)
    overwrite_formula(model, "hipocoa_c", "C34H48N7O20P3S")
    overwrite_charge(model, "hipocoa_c", -4)
    overwrite_charge(model, "hmbpp_c", -4) # pubchem C5H12O8P2 with charge 0; model=C5H8O8P2, so charge must be -4

    overwrite_charge(model, "istfrnA_e", -2)
    overwrite_formula(model, "istfrnA_e", "C17FeH19N2O14")
    overwrite_charge(model, "istfrnB_e", +1)
    overwrite_formula(model, "istfrnB_e", "C16FeH22N2O11")

    overwrite_charge(model, "lysglugly_c", 0)
    overwrite_charge(model, "lysglugly_e", 0)

    overwrite_charge(model, "man6pglyc_c", -3) # og = 0; alternative in bigg and in accordance with ecoli
    overwrite_charge(model, "mbhn_c", -1) # og = 0; https://modelseed.org/biochem/compounds/cpd15971
    overwrite_formula(model, "mcbtt_c", "C47H77N5O10") # was wrongly overwritten by a false bigg formula = [C43H71N5O10], metacyc also has the original one that was in the model
    overwrite_charge(model, "mcbtt_c", 0)
    overwrite_charge(model, "met_L_ala__L_c", -1)
    overwrite_charge(model, "met_L_ala__L_e", -1)
    overwrite_formula(model, "met_L_ala__L_c", "C8H15N2O3S")
    overwrite_formula(model, "met_L_ala__L_e", "C8H15N2O3S")
    overwrite_charge(model, "mhpglu_c", -4)
    overwrite_charge(model, "mi3p__D_c", -2) # og = 0, -2 according to bigg

    overwrite_formula(model, "Nforglu_c", "C6H7NO5")

    overwrite_charge(model, "ocACP_c", 0) # og = -1, 0 alterative in bigg and is in accordance with charge = 0 of ACP
    overwrite_formula(model, "ochol_c", "C27H42O2")
    overwrite_formula(model, "ocholc8coa_c", "C48H74N7O19P3S")
    overwrite_formula(model, "ocholc5coa_c", "C45H68N7O19P3S")
    overwrite_charge(model, "ocdcaACP_c", 0) # og = -1, 0 alterative in bigg and is in accordance with charge = 0 of ACP

    overwrite_charge(model, "phdcacoa_c", -4) # og=0 but its coa
    overwrite_charge(model, "phdca_c", -1) # og = 0 but https://modelseed.org/biochem/compounds/cpd16013
    overwrite_charge(model, "phdca_e", -1)
    overwrite_charge(model, "ppad_c", -2)
    overwrite_charge(model, "ptd1ino160_c", -1)
    overwrite_charge(model, "pqqh2_c", -3)
    overwrite_charge(model, "pqqh2_p", -3)
    overwrite_charge(model, "ppgpp_c", -6)
    overwrite_charge(model, "prepphth_c", -1) # og = 0; https://modelseed.org/biochem/compounds/cpd16028
    overwrite_charge(model, "prohisglu_c", -1) # og = -2; tripeptid pro-his-glu, only glu has -1 charge and other two are neutral
    overwrite_charge(model, "prohisglu_e", -1)

    overwrite_formula(model, "ribflv_c", "C17H20N4O6")
    overwrite_formula(model, "ribflv_e", "C17H20N4O6")

    # Salmochelin fixes (there are first fixes by Frowin in the apply mass balance function notebook)
    overwrite_formula(model, "salchsx_c", "C16H20NO11") # og C16H21NO11; https://pubchem.ncbi.nlm.nih.gov/compound/135397946
    overwrite_formula(model, "salchsx_e", "C16H20NO11")
    overwrite_formula(model, "salchsx_p", "C16H20NO11")
    overwrite_charge(model, "salchs2fe_c", 3) # to match salchs4fe
    overwrite_charge(model, "salchs2fe_p", 3)
    overwrite_charge(model, "salchs2fe_e", 3)
    #----- more S
    overwrite_charge(model, "salc_e", -1) # og = 0, -1 alterative in bigg
    overwrite_charge(model, "salc_c", -1)
    overwrite_charge(model, "scl_c", -7) # og = 0, -7 according to bigg
    overwrite_charge(model, "scys__L_c", -1)
    overwrite_charge(model, "ssaltpp_c", -3) # og = 0; 0 is not in bigg only -3 or -2
    overwrite_charge(model, "stfrnA_e", 0)
    overwrite_formula(model, "stfrnA_e", "C17H24N2O14")
    overwrite_charge(model, "stfrnA_c", 0)
    overwrite_formula(model, "stfrnA_c", "C17H24N2O14")
    overwrite_charge(model, "stfrnB_e", -2)
    #overwrite_formula(model, "stfrnB_e", "C16H22N2O11")
    overwrite_charge(model, "stfrnB_c", -2)
    #overwrite_formula(model, "stfrnB_c", "C16H22N2O11")

    overwrite_charge(model, "tag6p__D_c", -2)
    overwrite_charge(model, "tagdp__D_c", -4)
    overwrite_charge(model, "tamocta_c", -1) # og = 0; https://modelseed.org/biochem/compounds/cpd16038
    overwrite_charge(model, "tmhexc_c", -1) # og = 0; https://modelseed.org/biochem/compounds/cpd16050

    overwrite_charge(model, "udpacgal_c", -2) # og = 0, -2 alterative in bigg
    overwrite_charge(model, "udpacgal_p", -2) # og = 0; -2 alternative in bigg and in accordance with ecoli
    overwrite_charge(model, "udpacgal_e", -2)

    overwrite_charge(model, "vacc_c", -1)
    overwrite_charge(model, "vacc_p", -1)
    overwrite_charge(model, "vacc_e", -1)

    overwrite_charge(model, "xylan4_c", -1) # og = 0, no charge given in Bigg, but -1 fits equations
    overwrite_charge(model, "xylan4_e", -1)


    # second: reactions
    overwrite_reaction(model, "3HPAOX", # H was removed from this reaction
                       {"3hoxpac_c": -1.0,
                        "nadh_c": -1.0,
                        "o2_c": -1.0,
                        "34dhpha_c": 1.0,
                        "h2o_c": 1.0,
                        "nad_c": 1.0})

    overwrite_reaction(model, "3SALATAi", # this and ASPA2 are duplicate reactions (only differencs is an H), reaction was curated according to metacyc; https://biocyc.org/reaction?orgid=META&id=3-SULFINOALANINE-AMINOTRANSFERASE-RXN
                       {"3sala_c": -1.0,
                        "akg_c": -1.0,
                        "3snpyr_c": 1.0,
                        "glu__L_c": 1.0})

    overwrite_reaction(model, "ACOAM", # H was removed
                       {"ac_c": -1.0,
                        "atp_c": -1.0,
                        "acadl_c": 1.0,
                        "ppi_c": 1.0})

    overwrite_reaction(model, "ACPS1",
                       {"apoACP_c": -1.0,
                        "coa_c": -1.0,
                        "ACP_c": 1.0,
                        "pap_c": 1.0})

    overwrite_reaction(model, "ACPpds",
                       {"ACP_c": -1.0,
                        "h2o_c": -1.0,
                        "apoACP_c": 1.0,
                        "h_c": 2.0,
                        "pan4p_c": 1.0})

    overwrite_reaction(model, "ALDD31_1",
                       {"gly_c": 1.0,
                        "h_c": 2.0,
                        "h2o_c": -1.0,
                        "nad_c": -1.0,
                        "nadh_c": 1.0,
                        "amacald_c": -1})

    overwrite_reaction(model, "ASR",
                       {"aso4_c": -1.0,
                        "gthrd_c": -2.0,
                        "h_c": -1.0,
                        "aso3_c": 1.0,
                        "gthox_c": 1.0,
                        "h2o_c": 1.0})

    # https://biocyc.org/reaction?orgid=META&id=RXN-10737
    overwrite_reaction(model, "ASR2",
                       {"aso4_c": -1.0,
                        "trdrd_c": -1.0,
                        "h_c": -1.0,
                        "aso3_c": 1.0,
                        "h2o_c": 1.0,
                        "trdox_c": 1.0})

    # was overwritten by BIGG, but before that the H was in the reaction and equals also this reaction BKDC that is e.g. in AA1
    overwrite_reaction(model, "AT_MBD2",
                       {"dhlam_c": -1.0,
                        "ibcoa_c": -1.0,
                        "2mpdhl_c": 1.0,
                        "coa_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "BEF",
                       {"betald_c": -1.0,
                        "fad_c": -1.0,
                        "h2o_c": -1.0,
                        "fadh2_c": 1.0,
                        "glyb_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "CMLDC", # https://modelseed.org/biochem/reactions/rxn02483
                       {"4cml_c": -1.0,
                        "h_c": -1.0, # h changed from product to educt site
                        "5odhf2a_c": 1.0,
                        "co2_c": 1.0})
    if "4CMLCL_kt" not in model.reactions and "CMLDC" in model.reactions:
        rxn = model.reactions.get_by_id("CMLDC")
        rxn.id = "4CMLCL_kt"

    overwrite_reaction(model, "DACL", # https://biocyc.org/reaction?orgid=META&id=RXN0-5040 H was removed
                       {"abg4_c": -1.0,
                        "h2o_c": -1.0,
                        "4abz_c": 1.0,
                        "glu__D_c": 1.0})

    overwrite_reaction(model, "DHBZS2H",
                       {"23dhbzs2_c": -1.0,
                        "h2o_c": -1.0,
                        "h_c": 2.0, # new because of logic
                        "23dhbzs_c": 2.0})

     # https://biocyc.org/reaction?orgid=META&id=RXN-14477
    overwrite_reaction(model, "ENTERH",
                       {"enter_c": -1.0,
                        "h2o_c": -1.0,
                        "23dhbzs3_c": 1.0,
                        "h_c": 1.0}) # h was added

    overwrite_reaction(model, "FADD3",
                       {"atp_c": -1.0,
                        "coa_c": -1.0,
                        "hip_c": -1.0,
                        "hipcoa_c": 1.0,
                        "ppi_c": 1.0,
                        "amp_c": 1.0})

    overwrite_reaction(model, "FE3DHBZS3R",
                       {"fe3dhbzs3_c": -2.0,
                        "nadph_c": -1.0,
                        "23dhbzs3_c": 2.0,
                        "fe2_c": 2.0,
                        "h_c": 3.0,
                        "nadp_c": 1.0})

    overwrite_reaction(model, "FEDHBZS3R1",
                       {"fe3dhbzs3_c": -2.0,
                        "fadh2_c": -1.0,
                        "23dhbzs3_c": 2.0,
                        "fe2_c": 2.0,
                        "h_c": 4.0,
                        "fad_c": 1.0})

    overwrite_reaction(model, "FEDHBZS3R2",
                       {"fe3dhbzs3_c": -2.0,
                        "fmnh2_c": -1.0,
                        "23dhbzs3_c": 2.0,
                        "fe2_c": 2.0,
                        "h_c": 4.0,
                        "fmn_c": 1.0})

    overwrite_reaction(model, "FEDHBZS3R3",
                       {"fe3dhbzs3_c": -2.0,
                        "rbflvrd_c": -1.0,
                        "23dhbzs3_c": 2.0,
                        "fe2_c": 2.0,
                        "h_c": 4.0,
                        "ribflv_c": 1.0})

    overwrite_reaction(model, "FNOR",
                       {"fdxrd_c": -2.0,
                        "h_c": -1.0,
                        "nadp_c": -1.0,
                        fdxox_c: 2.0, # replaces fdxo_2_2_c
                        "nadph_c": 1.0})

    overwrite_reaction(model, "FORGLUIH2",
                       {"forglu_c": -1.0,
                        "h2o_c": -1.0,
                        "Nforglu_c": 1.0,
                        "nh4_c": 1.0})

    # https://biocyc.org/reaction?orgid=META&id=1.18.1.2-RXN change of stoichiometry
    overwrite_reaction(model, "FPRA",
                       {"fdxrd_c": -2.0,
                        "h_c": -1.0,
                        "nadp_c": -1.0,
                        fdxox_c: 2.0,
                        "nadph_c": 1.0})

    # https://modelseed.org/biochem/reactions/rxn28276 (immer noch charge imbalance, aber mass stimmt)
    overwrite_reaction(model, "GCDH",
                       {"glutcoa_c": -1.0,
                        "b2coa_c": 1.0,
                        "h_c": 1.0,
                        "co2_c": 1.0})

    # https://metacyc.org/reaction?orgid=META&id=GLUTAMATE-SYNTHASE-FERREDOXIN-RXN#
    overwrite_reaction(model, "GLMS_syn",
                       {"fdxrd_c": -2.0,
                        "akg_c": -1.0,
                        "gln__L_c": -1.0,
                        "h_c": -2.0,
                        "glu__L_c": 2.0,
                        fdxox_c: 2.0}) # replaces fdxo_2_2_c because we need +2 charge

    overwrite_reaction(model, "GLUTRS_3",
                       {"atp_c": -1.0,
                        "glu__L_c": -1.0,
                        "trnaglu_c": -1.0,
                        "amp_c": 1.0,
                        "glutrna_c": 1.0,
                        "ppi_c": 1.0})

    overwrite_reaction(model, "GLYCS_I",
                       {"gthrd_c": -1.0,
                        "mthgxl_c": -1.0,
                        "lgt__S_c": 1.0}) # og = lgt_s_c; they are duplicates

    overwrite_reaction(model, "GLYCS_II",
                       {"h2o_c": -1.0,
                        "lgt__S_c": -1.0, # og = lgt_s_c; they are duplicates
                        "gthrd_c": 1.0,
                        "h_c": 1.0,
                        "lac__L_c": 1.0})

    overwrite_reaction(model, "GLYTYRabc",
                       {"atp_c": -1.0,
                        "gly_tyr_e": -1.0,
                        "h2o_c": -1.0,
                        "adp_c": 1.0,
                        "gly_tyr_c": 1.0,
                        "pi_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "GLYLEUtr",
                       {"atp_c": -1.0,
                        "gly_leu_e": -1.0,
                        "h2o_c": -1.0,
                        "adp_c": 1.0,
                        "gly_leu_c": 1.0,
                        "pi_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "GLYPHEtr",
                       {"atp_c": -1.0,
                        "gly_phe_e": -1.0,
                        "h2o_c": -1.0,
                        "adp_c": 1.0,
                        "gly_phe_c": 1.0,
                        "pi_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "GLYCYSabc",
                       {"atp_c": -1.0,
                        "gly_cys_e": -1.0,
                        "h2o_c": -1.0,
                        "adp_c": 1.0,
                        "gly_cys_c": 1.0,
                        "pi_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "GTPDPK_1",
                       {"atp_c": -1.0,
                        "gtp_c": -1.0,
                        "amp_c": 1.0,
                        "gdptp_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "HSAC",
                       {"34dhsa_c": -1.0,
                        "o2_c": -1.0,
                        "49dsha_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "MECDPDH3_syn",
                       {"2mecdp_c": -1.0,
                        "fdxrd_c": -1.0,
                        "h_c": -1.0,
                        fdxox_c: 1.0, # replaces fdxo_2_2_c
                        "h2mb4p_c": 1.0,
                        "h2o_c": 1.0})
    overwrite_reaction(model, "MECDPDH4E", # AA3 and 1101 only have this reaction but not MECDPDH3_syn which in other models are duplicates; to be able to better compare between models, i am going to change the name of the reaction
                       {"2mecdp_c": -1.0,
                        "fdxrd_c": -1.0,
                        "h_c": -1.0,
                        fdxox_c: 1.0, # replaces fdxo_2_2_c
                        "h2mb4p_c": 1.0,
                        "h2o_c": 1.0})
    if "MECDPDH3_syn" not in model.reactions and "MECDPDH4E" in model.reactions:
        rxn = model.reactions.get_by_id("MECDPDH4E")
        rxn.id = "MECDPDH3_syn"

    overwrite_reaction(model, "MS_1",
                       {"hcys__L_c": -1.0,
                        "mhpglu_c": -1.0,
                        "hpglu_c": 1.0,
                        "met__L_c": 1.0})

    # there is still charge imbalance with his reaction but the fix gets rid of the mass inbalance
    overwrite_reaction(model, "NMO",
               {"etha_c": -1.0,
                "fmnh2_c": -1.0,
                "o2_c": -1.0,
                "acald_c": 1.0,
                "fmn_c": 1.0,
                "no2_c": 1.0,
                "h_c": 6.0  # this was og reaction but was overwritten with bigg info (H was lost)
                })

    overwrite_reaction(model, "OOR3r", # https://biocyc.org/reaction?orgid=META&id=2-OXOGLUTARATE-SYNTHASE-RXN; https://www.genome.jp/dbget-bin/www_bget?ec:1.2.7.3 EC number was given on BIGG page for that reaction but bigg was a bit off
                       {"akg_c": -1.0,
                        "coa_c": -1.0,
                        fdxox_c: -2.0,
                        "succoa_c": 1.0,
                        "co2_c": 1.0,
                        "h_c": 1.0,
                        "fdxrd_c": 2.0})

    overwrite_reaction(model, "PACPT_1",
                       {"amp_c": 1.0,
                        "coa_c": -1.0,
                        "ppcoa_c": 1.0,
                        "ppad_c": -1.0})

    # https://modelseed.org/biochem/reactions/rxn13395 out scl is only -7, in seed it is -8, so we need only one H
    overwrite_reaction(model, "PC2DHG",
                       {"dscl_c": -1.0,
                        "nadp_c": -1.0,
                        "nadph_c": 1.0,
                        "scl_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "PCADYOX", # https://modelseed.org/biochem/reactions/rxn01192
                       {"34dhbz_c": -1.0,
                        "o2_c": -1.0,
                        "CCbuttc_c": 1.0,
                        "h_c": 2.0}) # was added

    overwrite_reaction(model, "POR_syn",
                       {fdxox_c: -2.0, # replaces fdxo_2_2_c
                        "coa_c": -1.0,
                        "pyr_c": -1.0,
                        "accoa_c": 1.0,
                        "co2_c": 1.0,
                        "h_c": 1.0,
                        "fdxrd_c": 2.0})

    overwrite_reaction(model, "PRAIS",
                       {"atp_c": -1.0,
                        "fpram_c": -1.0,
                        "adp_c": 1.0,
                        "air_c": 1.0,
                        "pi_c": 1.0,
                        "h_c": 2.0})

    overwrite_reaction(model, "QSDH",
                       {"pqq_c": -1.0,
                        "skm_c": -1.0,
                        "3dhsk_c": 1.0,
                        "pqqh2_c": 1.0})

    # Salmochelin fixes (there are first fixes by Frowin in the apply mass balance function notebook)
    overwrite_reaction(model, "SALCHS1H",
                       {"h2o_c": -1.0,
                        "salchs1_c": -1.0,
                        "23dhbzs_c": 1.0,
                        "salchsx_c": 1.0,
                        "h_c": 2.0})
    overwrite_reaction(model, "SALCHS2H",
                       {"h2o_c": -1.0,
                        "salchs2_c": -1.0,
                        "salchs1_c": 1.0,
                        "salchsx_c": 1.0,
                        "h_c": 1.0})

    overwrite_reaction(model, "SMIA1",
                       {"fe3_e": -1.0,
                        "stfrnA_e": -1.0,
                        "istfrnA_e": 1.0})

    overwrite_reaction(model, "SMIA1abc",
                       {"atp_c": -1.0,
                        "h2o_c": -1.0,
                        "istfrnB_e": -1.0,
                        "adp_c": 1.0,
                        "fe3_c": 1.0,
                        "h_c": 1.0,
                        "pi_c": 1.0,
                        "stfrnB_c": 1.0})

    overwrite_reaction(model, "SMIA2abc",
                       {"atp_c": -1.0,
                        "h2o_c": -1.0,
                        "istfrnA_e": -1.0,
                        "adp_c": 1.0,
                        "fe3_c": 1.0,
                        "h_c": 1.0,
                        "pi_c": 1.0,
                        "stfrnA_c": 1.0})

    overwrite_reaction(model, "SMIB1",
                       {"fe3_e": -1.0,
                        "stfrnB_e": -1.0,
                        "istfrnB_e": 1.0})

    overwrite_reaction(model, "STAS",
                       {"atp_c": -2.0,
                       "h_c": -3.0, #H now acc to BIGG
                        "cit_c": -2.0,
                        "orn_c": -1.0,
                        "amp_c": 2.0,
                        "ppi_c": 2.0,
                        "stfrnA_c": 1.0,
                        })

    overwrite_reaction(model, "T6PK",
                       {"atp_c": -1.0,
                        "tag6p__D_c": -1.0,
                        "adp_c": 1.0,
                        "tagdp__D_c": 1.0,
                        "h_c": 1.0})

    # https://modelseed.org/biochem/reactions/rxn10816
    overwrite_reaction(model, "THZSN_1",
                       {"cys__L_c": -1.0,
                        "dxyl_c": -1.0,
                        fdxox_c: -1.0, # instead of fdx_2_2_c
                        "tyr__L_c": -1.0,
                        "4hba_c": 1.0,
                        "4mhetz_c": 1.0,
                        "co2_c": 1.0,
                        "fdxrd_c": 1.0,
                        "h2o_c": 1.0,
                        "h_c": 2.0, # 2 instead of 1;
                        "nh4_c": 1.0,
                        "pyr_c": 1.0})


    ## EMMA's fixes
    overwrite_charge(model, "fdxrd_c", 0) #acc to Frowins Maize work, https://www.metanetx.org/chem_info/MNXM178
    overwrite_charge(model, "fdxox_c", 2) #acc to Frowins Maize work

    overwrite_charge(model, "hexscoa_c", -4) #consistent now with acetyl coa charge
    overwrite_charge(model, "octscoa_c", -4) #consistent now with acetyl coa charge
    overwrite_charge(model, "octscoa_e", -4)#consistent now with acetyl coa charge
    overwrite_charge(model, "tetscoa_c", -4)#consistent now with acetyl coa charge
    overwrite_charge(model, "doscoa_c", -4)#consistent now with acetyl coa charge
    overwrite_charge(model, "eiscoa_c", -4)#consistent now with acetyl coa charge
    overwrite_charge(model, "m2butp_c", 0)
    overwrite_charge(model, "23dhbzs3_e", -1)
    overwrite_charge(model, "23dhbzs3_p", -1)

    delete_reaction(model, "IPPT") # deleted because this is a duplicate from reaction PPAKr with other (but wrong product), see http://pseudomonas.umaryland.edu/PAMDB?MetID=PAMDB001030 
    overwrite_formula(model, "23dhbzs3_e", "C30H29N3O16") #add one proton - see BIGG
    overwrite_formula(model, "23dhbzs3_p", "C30H29N3O16") #add one proton - see BIGG
    overwrite_formula(model, "3hcmrs7eACP_c", "C398H627O144N96P1S3") #og H626, now acc to BIGG 
    overwrite_formula(model, "3hbutACP_c", "C388H609N96O144P1S3") #og C4H5OSR -> replace R with actual stuff
    overwrite_formula(model, "5ohhip_c", "C13H18O4") #see BRENDA (https://www.brenda-enzymes.org/enzyme.php?ecno=6.2.1.41)
    overwrite_formula(model, "aad_c", "C12H14N5O8P") #og H16, aad only appears in AADa and AADb which are two part reactions of coenzyme A + acetate + ATP ↔ acetyl-CoA + AMP + diphosphate (BioCyc)., this is balanced
    overwrite_formula(model, "arachACP_c", "C404H641N96O143P1S3") #add proper ACP metabolites 
    overwrite_formula(model, "citdapp_c", "C9H14N2O9" ) #needs one O less 
    overwrite_formula(model, "ficytc_c", "C34H32FeN4O4") #og C33, BIGG and modelseed C34 (https://modelseed.org/biochem/compounds/cpd27756)
    overwrite_formula(model, "focytc_c", "C34H32FeN4O4") # og C33, BIGG C34, fixes both  'CYTBCYTC' and 'CYTBMQOR2pp', adds R4,
    overwrite_formula(model, "istfrnA_e", "C17FeH24N2O14") # before H17, now acc to bigg
    overwrite_formula(model, "istfrnA_c", "C17FeH24N2O14") # before H17, now acc to bigg
    overwrite_formula(model, "istfrnB_e", "C19FeH23N2O19") # before H17, now acc to bigg
    overwrite_formula(model, "istfrnB_c", "C19FeH23N2O19") # before H17, now acc to bigg
    overwrite_formula(model, "mcbtt_c", "C43H71N5O10") #now consistent with bigg 
    overwrite_formula(model, "nwharg_c","C6H14N4O3" ) #correct to H14 
    overwrite_formula(model, "phdcaACP_c", "C407H639N96O144PS3") #add proper ACP formula 
    overwrite_formula(model, "prephthACP_c", "C416H663N96O146PS3" ) #add proper ACP formula
    overwrite_formula(model, "prepphthACP_c", "C419H661N96O147PS3") # add proper ACP formula 

    overwrite_formula(model, "stfrnB_c", "C19H23N2O19" )
    overwrite_formula(model, "stfrnB_e", "C19H23N2O19" )
    overwrite_formula(model, "uaccg_e", "C20H29N3O19P2") #instead of H26, in accordance to http://bigg.ucsd.edu/models/iCN900/metabolites/uaccg_e

    overwrite_reaction(model, "4HDPROO", {
       "4hpro_DC_c": -1.0,
        "ficytc_c": -2.0,
        "1py4h3c_c": 1.0,
        "focytc_c": 2.0,
        "h_c": 3.0 #added two more protons, consistent with all other ficyct_c -> focyct_c reactions
    } )
    overwrite_reaction(model, "ACKILE", {
        "adp_c": -1.0,
        m2butp_c: -1.0,
        "2mba_c": 1.0,
        "atp_c": 1.0
    }) 

    overwrite_reaction(model, "ALDD19xr", {
        "h2o_c": -1.0,
        "nad_c": -1.0,
        "pacald_c": -1.0,
        "h_c": 2.0,
        "nadh_c": 1.0,
        "pac_c": 1.0
    }) 
    overwrite_reaction(model, "BTS_1", {
        "dtbt_c": -1.0, 
        "s_c": -2.0,
        "h2s_c": 1.0,
        "btn_c": 1.0 #remove proton here -> now consistent with other BTS reactions
    })
    overwrite_reaction(model, "CCPpp", {
        "focytc_c": -2.0,
        "h2o2_p": -1.0,
        "h_p": -2.0, #add as h2o2 + 2h -> 2 h20
        "ficytc_c": 2.0,
        "h2o_p": 2.0
    })
    overwrite_reaction(model, "CITDAPPS", {
    "23dappa_c": -1.0, "atp_c": -1.0, "cit_c": -1.0, "h2o_c": -1.0, 
    "adp_c": 1.0, "citdapp_c": 1.0, "pi_c": 1.0, "h_c": 1.0
    })
    overwrite_reaction(model, "CO2FO", {
        "co2_c": -1.0,
        "h_c": -2.0,
        "fdxrd_c": -1.0,
        "h2o_c": 1.0,
        "co_c": 1.0,
        fdxox_c: 1.0 #replaces fdxo_2_2_c
    })
    if "CYO1b" not in model.reactions and "CYTCAA3pp" in model.reactions: #AA and BB duplicates with wrong formula (https://iubmb.qmul.ac.uk/enzyme/EC7/1/1/9.html): 2 ferrocytochrome c +0.5 O2 + 4 H+[side 1] = 2 ferricytochrome c + H2O + 2 H+[side 2] -> overwrite AA with BB as there are more models containing BB 
        rxn = model.reactions.get_by_id("CYTCAA3pp")
        rxn.id = "CYO1b"
    overwrite_reaction(model, "CYO1b", {
        "focytc_c": -2.0,
        "o2_c": -0.5,
        "h_c": -2.0, #add as h2o2 + 2h -> 2 h20
        "ficytc_c": 2.0,
        "h2o_c": 1.0
    })

    if "CYO1b" not in model.reactions and "CYTCBB3pp" in model.reactions: #AA and BB duplicates with wrong formula (https://iubmb.qmul.ac.uk/enzyme/EC7/1/1/9.html): 2 ferrocytochrome c +0.5 O2 + 4 H+[side 1] = 2 ferricytochrome c + H2O + 2 H+[side 2] -> overwrite AA with BB as there are more models containing BB 
        rxn = model.reactions.get_by_id("CYTCBB3pp")
        rxn.id = "CYO1b"
    overwrite_reaction(model, "CYO1b", {
        "focytc_c": -2.0,
        "o2_c": -0.5,
        "h_c": -2.0, #add as h2o2 + 2h -> 2 h20
        "ficytc_c": 2.0,
        "h2o_c": 1.0
    })
    if "CYO1b" in model.reactions and "CYTCBB3pp" in model.reactions: #CYTCBB3pp is unbalanced version of Cy01 and only found in one model. Acc. to BioCyc, the reaction should be +H (educts, https://biocyc.org/ECOLI/NEW-IMAGE?type=EC-NUMBER&object=EC-2.7.7.2), replace for comparability
        delete_reaction(model, "CYTCBB3pp")
    if "CYO1b" in model.reactions and "CYTCAA3pp" in model.reactions: #CYTCBB3pp is unbalanced version of Cy01 and only found in one model. Acc. to BioCyc, the reaction should be +H (educts, https://biocyc.org/ECOLI/NEW-IMAGE?type=EC-NUMBER&object=EC-2.7.7.2), replace for comparability
        delete_reaction(model, "CYTCAA3pp")
    overwrite_reaction(model, "CYO1_KT",{ #remove hydrogen atom from educts: quinol + 2 ferricytochrome c = quinone + 2 ferrocytochrome c + 2 H+[side 2] (https://iubmb.qmul.ac.uk/enzyme/EC7/1/1/8.html)
        "q8h2_c": -1.0,
        "ficytc_c": -2.0,
        "q8_c": 1.0,
         "h_p": 2.0,
        "focytc_c": 2.0
    })

    overwrite_reaction(model, "CYO4pp", {
        "ficytc_c": -2.0,
        "pqqh2_p": -1.0,
        "focytc_c": 2.0,
        "pqq_p": 1.0,
        "h_p": 2.0 #add two hydrogen atoms based on logic of oxidation of pqqh2 to ppq 
    })

    '''if "CYTCBB3pp" not in model.reactions and "CYTCAA3pp" in model.reactions: #AA and BB duplicates with wrong formula (https://iubmb.qmul.ac.uk/enzyme/EC7/1/1/9.html): 2 ferrocytochrome c +0.5 O2 + 4 H+[side 1] = 2 ferricytochrome c + H2O + 2 H+[side 2] -> overwrite AA with BB as there are more models containing BB 
        rxn = model.reactions.get_by_id("CYTCAA3pp")
        rxn.id = "CYTCBB3pp"
    overwrite_reaction(model, "CYTCBB3pp", # correct formula
                       {"h_c": -4.0,
                        "o2_c": -0.5,
                        "focytc_c": -2.0,
                        "h2o_c": 1.0,
                        "h_p": 2.0,
                        "ficytc_c": 2.0
    })'''

    if "FMNAT" not in model.reactions and "FMNAT_1" in model.reactions: #FMNAT_1 is unbalanced version of FMNAT and only found in one model. Acc. to BioCyc, the reaction should be +H (educts, https://biocyc.org/ECOLI/NEW-IMAGE?type=EC-NUMBER&object=EC-2.7.7.2), replace for comparability
        rxn = model.reactions.get_by_id("FMNAT_1")
        rxn.id = "FMNAT"
    overwrite_reaction(model, "FMNAT", # add extra hydrogen atom to educts 
                {"atp_c": -1.0,
                "fmn_c": -1.0,
                "h_c": -1.0, #this is the extra hydrogen atom 
                "fad_c": 1.0,
                "ppi_c": 1.0})
    if "FMNAT" in model.reactions and "FMNAT_1" in model.reactions: #FMNAT_1 is unbalanced version of FMNAT and only found in one model. Acc. to BioCyc, the reaction should be +H (educts, https://biocyc.org/ECOLI/NEW-IMAGE?type=EC-NUMBER&object=EC-2.7.7.2), replace for comparability
        delete_reaction(model, "FMNAT_1")
    overwrite_reaction(model, "FNOR", #products were missing 
                       {"h_c": -1.0,
                         "nadp_c": -1.0,
                        "fdxrd_c": -2.0,
                        "nadph_c" : 1.0,
                        fdxox_c: 2.0 #replaces fdxo_2_2_c
                       })
    
    overwrite_reaction(model, "FNRR3", {
            "h2_c": -1.0,
            fdxox_c: -1.0, #replaces fdxo_2_2_c
            "fdxrd_c": 1.0,
            "h_c": 2.0,
        })
    if "GAPD" in model.reactions and "GAPD_1" in model.reactions: #GAPD_1 is unbalanced version of GAPD and only found in one model. Acc. to MetaNEtx, the reaction should be +H (educts,https://www.metanetx.org/equa_info/MNXR100040), replace for comparability
        delete_reaction(model, "GAPD_1")
    if "GLUTRS" in model.reactions and "GLUTRS_2" in model.reactions: #GLUTRS_2 is unbalanced version of GLUTRS and only found in one model. Acc. to BioCyc, the reaction should be +H (educts, https://biocyc.org/ECOLI/NEW-IMAGE?type=EC-NUMBER&object=EC-2.7.7.2), replace for comparability
        delete_reaction(model, "GLUTRS_2")
    overwrite_reaction(model, "GLYCS_I",{
                        "gthrd_c": -1.0,
                        "mthgxl_c": -1.0,
                        "lgt_S_c": 1.0 #add product as in BIGG 
                        })
    overwrite_reaction(model, "GLUFT", {
        "5fthf_c": -1.0,
        "glu__L_c": -1.0,
        "h_c": 1.0, # add one proton to the product side and remove the proton from the educt side (see https://www.metanetx.org/equa_info/MNXR100282)
        "thf_c": 1.0,
        "Nforglu_c": 1.0
        })
    if "HEMEOS" in model.reactions and "HEMEOS_1" in model.reactions: #HEMEOS_1 is unbalanced version of HEMEOS and only found in one model. Acc. to BioCyc, the reaction should be +H (educts, https://biocyc.org/ECOLI/NEW-IMAGE?type=EC-NUMBER&object=EC-2.7.7.2), replace for comparability
        delete_reaction(model, "HEMEOS_1")
    if "HEMEOS" not in model.reactions and "HEMEOS_1" in model.reactions: #HEMEOS_1 is unbalanced version of HEMEOS 
        rxn = model.reactions.get_by_id("HEMEOS_1")
        rxn.id = "HEMEOS"
    overwrite_reaction(model, "HEMEOS",
                       {"frdp_c": -1.0,
                        "h2o_c": -1.0,
                        "pheme_c": -1.0,
                        "hemeO_c": 1.0,
                        "ppi_c": 1.0})
    overwrite_reaction(model, "HIPA", { #remove proton from educts 
        "hip_c": -1.0,
        "nadh_c": -1.0,
        "5ohhip_c" : 1.0,
        "nad_c": 1.0
    })
    overwrite_reaction(model, "IPDAB", {
        "atp_c": -1.0,
        "coa_c": -1.0,
        "h_c": -1.0, #correct from 3H to 1H (https://www.brenda-enzymes.org/enzyme.php?ecno=6.2.1.41)
        "5ohhip_c" : -1.0,
        "amp_c": 1.0,
        "ppi_c": 1.0,
        "5ohhipcoa_c": 1.0
    })
    overwrite_reaction(model, "L_LACD",
                       {"lac__L_c": -1.0,
                        "ficytc_c": -2.0,
                        "pyr_c": 1.0,
                        "focytc_c": 2.0,
                        "h_c": 2.0, #added two hydrogen atoms to products (https://www.brenda-enzymes.org/structure.php?show=reaction&id=33683&type=I&displayType=marvin)
                        })

    if "MAN6Gpts" in model.reactions: ##MAN6Gpts uses PTS system without phosphorylating the sugar, needs to be phosphorylated, see https://smpdb.ca/pathwhiz/pathways/PW000786) -> replace with correct reaction 
        rxn = model.reactions.get_by_id("MAN6Gpts")
        if "MANGLYCptspp" not in model.reactions:
            rxn = model.reactions.get_by_id("MAN6Gpts")
            rxn.id = "MANGLYCptspp"
        else:
            delete_reaction(model, "MAN6Gpts")

    if "MANGLYCptspp" in model.reactions:
        add_new_met(model,"manglyc_e", formula="C9H15O9", name="2-O-alpha-mannosyl-D-glycerate", charge=-1, compartment="e")
        add_new_met(model,"manglyc_p", formula="C9H15O9", name="2-O-alpha-mannosyl-D-glycerate", charge=-1, compartment="p")
        add_new_rxn(model, "MANGLYCtex", "2-O-alpha-mannosyl-D-glycerate transport via diffusion (extracellular to periplasm)", -1000, 1000, {"manglyc_e": -1.0, "manglyc_p": 1.0})
        overwrite_reaction(
            model, 
            "MANGLYCptspp",
            {
                "pep_c": -1.0,
                "manglyc_p": -1.0,
                "man6pglyc_c": 1.0,
                "pyr_c": 1.0
            }
        )    
    overwrite_reaction(model, "MRF",
                       {"h_c" : -3.0,
                        "mlthf_c" : -1.0,
                        "fdxrd_c" : -1.0,
                        "5mthf_c" : 1.0,
                        fdxox_c : 1.0}) #replaces fdxo_2_2_c
    if "MI3PP" not in model.reactions and "MI3PP_1" in model.reactions: #MI3PP_1 is unbalanced version of MI3PP, MI3PP does not have protons in products 
        rxn = model.reactions.get_by_id("MI3PP_1")
        rxn.id = "MI3PP"
    overwrite_reaction(model, "MI3PP",
                       {"h2o_c": -1.0,
                        "mi3p__D_c": -1.0,
                        "inost_c": 1.0,
                        "pi_c": 1.0})
    if "NAR_syn" not in model.reactions and "NAR_syn_2" in model.reactions: #same reaction but reverse 
        rxn = model.reactions.get_by_id("NAR_syn_2")
        rxn.id = "NAR_syn"
        model.reactions.NAR_syn.bounds = (-1000, 1000) #fuse two reactions
    overwrite_reaction(model, "NAR_syn", # correct formula
                       {"h_c": -2.0,
                        "fdxrd_c": -2.0,
                        "no3_c": -1.0,
                        "h2o_c": 1.0,
                        fdxox_c: 2.0, #replace fdxo_2_2_c
                        "no2_c": 1.0})
    if "NAR_syn" in model.reactions and "NAR_syn_2" in model.reactions: #NAR_syn_2 is unbalanced version of NAR_syn and only found in one model
        delete_reaction(model, "NAR_syn_2")
    overwrite_reaction(model, "NFORGLUAH", {
        "h2o_c": -1.0,
        "Nforglu_c": -1.0,
        "for_c": 1.0, 
        "glu__L_c": 1.0
    }) #remove protons from product side (https://biocyc.org/reaction?orgid=META&id=N-FORMYLGLUTAMATE-DEFORMYLASE-RXN)

    overwrite_reaction(model, "NOR_syn_1", {
        "fdxrd_c": -6.0, 
        "h_c": -7.0, 
        "no2_c": -1.0,
        "h2o_c": 2.0,
        fdxox_c: 6.0, #replace fdxo_2_2_c
        "nh3_c": 1.0 })
    overwrite_reaction(model, "PENAM", {
        "h_p": 1.0, #add proton ot balance (https://modelseed.org/biochem/reactions/rxn02871)
                        "h2o_p": -1.0,
                        "peng_p": -1.0,
                        "6apa_p": 1.0,
                        "pac_p": 1.0})
    if "PLPS" not in model.reactions and "PLPS_1" in model.reactions: #PLPS_1 is unbalanced version of PLPS, PLPS has one hydrogen more in products 
        rxn = model.reactions.get_by_id("PLPS_1")
        rxn.id = "PLPS"
    overwrite_reaction(model, "PLPS",
                       {"g3p_c": -1.0,
                        "gln__L_c": -1.0,
                        "r5p_c": -1.0,
                        "glu__L_c": 1.0,
                        "h2o_c": 3.0,
                        "h_c": 1.0,
                        "pydx5p_c": 1.0,
                        "pi_c": 1.0})
    
    if "PRAIS" not in model.reactions and "PRAIS_1" in model.reactions: #PRAIS_1 is unbalanced version of PRAIS, PRAIS has one hydrogen more in products 
        rxn = model.reactions.get_by_id("PRAIS_1")
        rxn.id = "PRAIS"
    overwrite_reaction(model, "PRAIS",
                       {"atp_c": -1.0,
                        "fpram_c": -1.0,
                        "adp_c": 1.0,
                        "air_c": 1.0,
                        "h_c": 2.0,
                        "pi_c": 1.0})
    if "PRAIS" in model.reactions and "PRAIS_1" in model.reactions: #PRAIS_1 is unbalanced version of PRAIS and only found in one model
        delete_reaction(model, "PRAIS_1")

    overwrite_reaction(model, "PRFGCL",
        {"atp_c": -1.0,
                        "fpram_c": -1.0,
                        "adp_c": 1.0,
                        "air_c": 1.0,
                        "h_c": 2.0, #add one proton (https://www.metanetx.org/equa_info/MNXR138720)
                        "pi_c": 1.0}) 
    if "PRFGS" in model.reactions and "PRFGS_1" in model.reactions: #PRFGS_1 is unbalanced version of PRFGS and only found in one model
        delete_reaction(model, "PRFGS_1")
    if "PRFGS" not in model.reactions and "PRFGS_1" in model.reactions: #PRFGS is unbalanced version of PRFGS 
        rxn = model.reactions.get_by_id("PRFGS_1")
        rxn.id = "PRFGS"
    overwrite_reaction(model, "PRFGS",
                       {"atp_c": -1.0,
                        "fgam_c": -1.0,
                        "gln__L_c": -1.0,
                        "h2o_c": -1.0,
                        "adp_c": 1.0,
                        "fpram_c": 1.0,
                        "glu__L_c": 1.0,
                        "h_c": 1.0,
                        "pi_c": 1.0})
    if "PSP" not in model.reactions and "PSP_1" in model.reactions: #PSP_1 is unbalanced version of PSP 
        rxn = model.reactions.get_by_id("PSP_1")
        rxn.id = "PSP"
    overwrite_reaction(model, "PSP",
                       {"atp_c": -1.0,
                        "fpram_c": -1.0,
                        "adp_c": 1.0,
                        "air_c": 1.0,
                        "h_c": 2.0,
                        "pi_c": 1.0})
    overwrite_reaction(model, "PTAILE", {
        "2mbcoa_c": -1.0,
        "pi_c": -1.0,
        m2butp_c: 1.0,
        "coa_c": 1.0
    }) 

    
    if "RBFK" not in model.reactions and "RBFK_1" in model.reactions: #RBFK_1 is unbalanced version of RBFK and only found in one model. Acc. to BioCyc, the reaction should be +H (https://biocyc.org/reaction?orgid=ECOLI&id=RIBOFLAVINKIN-RXN), replace for comparability
        rxn = model.reactions.get_by_id("RBFK_1")
        rxn.id = "RBFK"
    overwrite_reaction(model, "RBFK", # add extra hydrogen atom to products 
                {"atp_c": -1.0,
                "ribflv_c": -1.0,
                "h_c": 1.0, #this is the extra hydrogen atom 
                "adp_c": 1.0,
                "fmn_c": 1.0})
    if "RBFK" in model.reactions and "RBFK_1" in model.reactions: #RBFK_1 is unbalanced version of RBFK and only found in one model
        delete_reaction(model, "RBFK_1")
    overwrite_reaction(model, "TAGabc", {
                        "atp_c": -1.0,
                        "tagur_e": -1.0,
                        "h2o_c": -1.0, #add water to ABC transporter
                        "adp_c": 1.0,
                        "tagur_c": 1.0,
                        "h_c": 1.0, #only one proton 
                        "pi_c": 1.0
    })
    overwrite_reaction(model, "UNK2", {
        "gln__L_c": -1.0,
        "h_c": -2.0, #should be two protons https://modelseed.org/biochem/reactions/rxn10944
        "2kmb_c": -1.0,
        "glu__L_c": 1.0,
        "met__L_c": 1.0
    })
    add_metabolites(model, "MRF", {fdxox_c: 1.0})
    add_metabolites(model, "FNOR", {fdxox_c: 2.0})
    add_metabolites(model, "GLYCS_I", {lgt__S_c: 1.0})
    add_metabolites(model, "MANGLYCtex" , {manglyc_p: 1.0})
    #add_metabolites(model, "MANGLYCptspp" , {manglyc_p: 1.0})

## Main

In [17]:
imbalances_after_overwrite = {}
for file in os.listdir(model_dir):
        if not file.endswith(('.xml')):
            continue
        if f'{file[:-4]}_or_mb1.xml' not in os.listdir(save_dir):
            model = read_sbml_model(model_dir+file)
            overwrite_with_BiGG_metabolites(model)
            overwrite_with_BIGG_reactions(model)
            overwrite_manual(model)
            unbalanced_rxns = check_balance(model, print_results=False)
            unbalanced_rxns = [r.id for r in unbalanced_rxns]
            imbalances_after_overwrite[file] = unbalanced_rxns

            write_sbml_model(model, save_dir+f'{file[:-4]}_or_mb1.xml')

Adding exchange reaction EX_14glucan_e with default bounds for boundary metabolite: 14glucan_e.
Adding exchange reaction EX_23camp_e with default bounds for boundary metabolite: 23camp_e.
Adding exchange reaction EX_23ccmp_e with default bounds for boundary metabolite: 23ccmp_e.
Adding exchange reaction EX_23cgmp_e with default bounds for boundary metabolite: 23cgmp_e.
Adding exchange reaction EX_23cump_e with default bounds for boundary metabolite: 23cump_e.
Adding exchange reaction EX_25dkglcn_e with default bounds for boundary metabolite: 25dkglcn_e.
Adding exchange reaction EX_2ameph_e with default bounds for boundary metabolite: 2ameph_e.
Adding exchange reaction EX_2dhglcn_e with default bounds for boundary metabolite: 2dhglcn_e.
Adding exchange reaction EX_2m35mdntha_e with default bounds for boundary metabolite: 2m35mdntha_e.
Adding exchange reaction EX_34dhbz_e with default bounds for boundary metabolite: 34dhbz_e.
Adding exchange reaction EX_35dnta_e with default bounds for b

m_1350_ does not contain one of the metabolites
m_1350_ does not contain one of the metabolites


Adding exchange reaction EX_23dappa_e with default bounds for boundary metabolite: 23dappa_e.
Adding exchange reaction EX_25dkglcn_e with default bounds for boundary metabolite: 25dkglcn_e.
Adding exchange reaction EX_2ameph_e with default bounds for boundary metabolite: 2ameph_e.
Adding exchange reaction EX_2dhglcn_e with default bounds for boundary metabolite: 2dhglcn_e.
Adding exchange reaction EX_2m35mdntha_e with default bounds for boundary metabolite: 2m35mdntha_e.
Adding exchange reaction EX_2pglyc_e with default bounds for boundary metabolite: 2pglyc_e.
Adding exchange reaction EX_34dhpac_e with default bounds for boundary metabolite: 34dhpac_e.
Adding exchange reaction EX_35dnta_e with default bounds for boundary metabolite: 35dnta_e.
Adding exchange reaction EX_3amp_e with default bounds for boundary metabolite: 3amp_e.
Adding exchange reaction EX_3cmp_e with default bounds for boundary metabolite: 3cmp_e.
Adding exchange reaction EX_3gmp_e with default bounds for boundary me

## Assessment after fixes

In [18]:
unique_reactions = {react for reacts_list in imbalances_after_overwrite.values() for react in reacts_list}

In [ ]:
for modid, reacts in imbalances_after_overwrite.items():
    extension = modid.split(".")
    name2load = extension[0] + "_or_mb1.xml"
    mod = read_sbml_model(save_dir + name2load)

    print(modid)
    for r in reacts:
        reac = mod.reactions.get_by_id(r)
        balance = reac.check_mass_balance()
        print(r, balance) 
    print("xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

2862.xml
ALDD19xr {'charge': -1.0}
AMMQT8 {'charge': -2.0}
COCO2 {'charge': -2.0}
CYO1_KT {'charge': 2.0}
CYO1b {'H': 2.0}
CYTCAA3pp {'H': 2.0}
CYTCBB3pp {'charge': -2.0}
FNOR {'charge': 2.0}
GCDH {'charge': 2.0}
KGD2 {'charge': 2.0}
NAR_syn {'charge': -2.0, 'X': 2.0}
NAR_syn_2 {'charge': 2.0, 'X': -1.0}
NMO {'charge': 4.0}
PACCOAL {'H': -1.0}
PACOAT {'H': 1.0}
PRFGCL {'charge': -1.0, 'H': -1.0}
TAGabc {'charge': 1.0, 'H': 1.0}
XYLe {'charge': -2.0}
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
796.xml
ALDD19xr {'charge': -1.0}
AMID2 {'H': 1.0}
AMMQT8 {'charge': -2.0}
BTS2 {'charge': 2.0}
COCO2 {'charge': -2.0}
CYO1_KT {'charge': 2.0}
CYO1b {'H': 2.0}
CYO4pp {'charge': 2.0}
CYTCAA3pp {'H': 2.0}
CYTCBB3pp {'charge': -2.0}
DHBSZ3FEabcpp {'charge': 3.0}
GCDH {'charge': 2.0}
GLMS_syn {'charge': 2.0}
H2CD {'charge': -1.0}
H2CT {'charge': 4.0}
HDECH {'charge': 2.0}
HMEDS {'charge': 2.0}
ICCT {'charge': -3.0}
NMO {'charge': 4.0}
PACL {'H': -1.0}
PENAM {

In [28]:
def analyze_directory_imbalances(model_dir, output_csv="mass_imbalances_5thiter.csv"):
    """
    Loads all SBML (.xml) models in model_dir, accumulates all reactions 
    with elemental mass imbalances, and saves them to a CSV file.
    """
    all_imbalances = []
    
    # List all XML files in the directory
    model_files = [f for f in os.listdir(model_dir) if f.endswith('.xml')]
    
    if not model_files:
        print(f"No .xml files found in {model_dir}")
        return pd.DataFrame()

    for file_name in model_files:
        file_path = os.path.join(model_dir, file_name)
        print(f"Processing {file_name}...")
        
        try:
            # Load the model
            model = read_sbml_model(file_path)
        except Exception as e:
            print(f"Error reading {file_name}: {e}")
            continue

        for rxn in model.reactions:
            # Skip boundary reactions (exchange, demand, sink) and growth 
            if rxn.boundary:
                continue
            if "Growth" in rxn.id:
                continue
                
            balance_errors = rxn.check_mass_balance()
            
            if balance_errors:
                # Isolate mass (elemental) imbalance by filtering out 'charge'
                mass_elements = {k: v for k, v in balance_errors.items() if k != 'charge'}
                
                # If mass_elements is not empty, we have a mass imbalance
                if mass_elements:
                    all_imbalances.append({
                        'Model_File': file_name,
                        'Model_ID': model.id,
                        'Reaction_ID': rxn.id,
                        'Reaction_Name': rxn.name,
                        'Reaction_String': rxn.reaction,
                        'Mass_Imbalance': str(mass_elements),
                        'Charge_Imbalance': balance_errors.get('charge', 0)
                    })

    # Convert to DataFrame
    df = pd.DataFrame(all_imbalances)
    
    # Save to file if data was found
    if not df.empty:
        df.to_csv(output_csv, index=False)
        print(f"\nSuccess! Found {len(df)} mass-imbalanced reactions across your models.")
        print(f"Saved results to '{output_csv}'")
    else:
        print("\nNo mass-imbalanced reactions found in any of the models.")
        
    return df


In [29]:
df_imbalances = analyze_directory_imbalances(save_dir)

Processing 1174_or_mb1.xml...
Processing 1124_or_mb1.xml...
Processing 459_or_mb1.xml...
Processing 978_or_mb1.xml...
Processing 2862_or_mb1.xml...
Processing 2774_or_mb1.xml...
Processing 644_or_mb1.xml...
Processing 161_or_mb1.xml...
Processing 895_or_mb1.xml...
Processing 946_or_mb1.xml...
Processing 1334_or_mb1.xml...
Processing 1252_or_mb1.xml...
Processing 1114_or_mb1.xml...
Processing 364_or_mb1.xml...
Processing 868_or_mb1.xml...
Processing 1101_or_mb1.xml...
Processing 947_or_mb1.xml...
Processing 892_or_mb1.xml...
Processing 1432_or_mb1.xml...
Processing 761_or_mb1.xml...
Processing 1167_or_mb1.xml...
Processing 790_or_mb1.xml...
Processing 1208_or_mb1.xml...
Processing 1362_or_mb1.xml...
Processing 163_or_mb1.xml...
Processing 397_or_mb1.xml...
Processing 1234_or_mb1.xml...
Processing 1357_or_mb1.xml...
Processing 100_or_mb1.xml...
Processing 796_or_mb1.xml...
Processing 709_or_mb1.xml...
Processing 352_or_mb1.xml...
Processing 1350_or_mb1.xml...
Processing 778_or_mb1.xml...

#### Check if all models grow post mass balancing

In [24]:
model_dir = "/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1/"

In [25]:
gd = {}
for file in os.listdir(model_dir):
        if not file.endswith(('.xml')):
            continue
        model = read_sbml_model(model_dir+file)
        sol = model.slim_optimize()
        gd[model.id] = sol